## 머신러닝 재설계
미국에서 잘 팔린 상품들의 피처 패턴이 무엇인지 학습
기존 회귀 --> 분류 모델로 재설계

### 베이스라인 검토

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

files = {
    "cleanser": r"C:\workspace\finalproject\data\model_data_v1\model_cleanser",
    "skincare": r"C:\workspace\finalproject\data\model_data_v1\model_skincare",
    "mask":     r"C:\workspace\finalproject\data\model_data_v1\model_mask",
    "suncare":  r"C:\workspace\finalproject\data\model_data_v1\model_suncare",
}

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "RandomForest":       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "XGBoost":            XGBClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbosity=0, n_jobs=-1),
    "LightGBM":           LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, n_jobs=-1, verbose=-1),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for cat, path in files.items():
    X = pd.read_csv(f"{path}_X.csv")
    y_raw = pd.read_csv(f"{path}_y.csv").squeeze()
    
    # 상위 30% → 2, 30~70% → 1, 하위 30% → 0
    y = pd.cut(y_raw,
               bins=[0, y_raw.quantile(0.3), y_raw.quantile(0.7), 1.0],
               labels=[0, 1, 2],
               include_lowest=True
    ).astype(int)
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    print(f"\n{'='*55}")
    print(f"[{cat.upper()}] 클래스 분포: 0={( y==0).sum()} / 1={(y==1).sum()} / 2={(y==2).sum()}")
    print(f"{'='*55}")
    print(f"{'모델':<22} {'Accuracy':>10} {'F1(macro)':>12}")
    print(f"{'-'*55}")
    
    for name, model in models.items():
        acc = cross_val_score(model, X_scaled, y, cv=skf, scoring='accuracy').mean()
        f1  = cross_val_score(model, X_scaled, y, cv=skf, scoring='f1_macro').mean()
        print(f"{name:<22} {acc:>10.4f} {f1:>12.4f}")

c:\Users\asiae\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\asiae\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (



[CLEANSER] 클래스 분포: 0=364 / 1=484 / 2=364
모델                       Accuracy    F1(macro)
-------------------------------------------------------
LogisticRegression         0.3862       0.3592
RandomForest               0.3986       0.3778
XGBoost                    0.3953       0.3761
LightGBM                   0.4076       0.3908

[SKINCARE] 클래스 분포: 0=1324 / 1=1764 / 2=1324
모델                       Accuracy    F1(macro)
-------------------------------------------------------
LogisticRegression         0.3998       0.3499
RandomForest               0.3830       0.3733
XGBoost                    0.3946       0.3657
LightGBM                   0.3826       0.3701

[MASK] 클래스 분포: 0=368 / 1=490 / 2=368
모델                       Accuracy    F1(macro)
-------------------------------------------------------
LogisticRegression         0.4111       0.3942
RandomForest               0.4005       0.3847
XGBoost                    0.3988       0.3830
LightGBM                   0.3834       0.3727

[

In [2]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import numpy as np

path = r"C:\workspace\finalproject\data\model_data_v1\model_cleanser"
X = pd.read_csv(f"{path}_X.csv")
y_raw = pd.read_csv(f"{path}_y.csv").squeeze()

y = pd.cut(y_raw,
           bins=[0, y_raw.quantile(0.3), y_raw.quantile(0.7), 1.0],
           labels=[0, 1, 2],
           include_lowest=True
).astype(int)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_scaled, y)

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

print(importance_df.head(20).to_string(index=False))

                      feature  importance
      benefit_norm__hydrating    0.056020
         material_norm__vegan    0.053918
    skin_type_norm__sensitive    0.053733
  material_norm__paraben_free    0.050737
  material_norm__cruelty_free    0.050571
       benefit_norm__soothing    0.048954
          skin_type_norm__all    0.044479
     benefit_norm__anti_aging    0.042810
    benefit_norm__exfoliating    0.041319
          skin_type_norm__dry    0.040998
    benefit_norm__brightening    0.039141
material_norm__fragrance_free    0.035555
        item_form_norm__cream    0.035526
         skin_type_norm__oily    0.034652
          item_form_norm__gel    0.031844
  skin_type_norm__combination    0.031101
          item_form_norm__oil    0.031063
  material_norm__sulfate_free    0.030454
       skin_type_norm__normal    0.029395
      material_norm__oil_free    0.027244


### y값 재정규화

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.decomposition import PCA

files = ['cleanser', 'mask', 'skincare', 'suncare']

for name in files:
    df = pd.read_csv(rf'C:\workspace\finalproject\data\model_data_v1\model_{name}_final.csv')
    
    # y1, y2 스케일링
    scaler = StandardScaler()
    y_scaled = scaler.fit_transform(df[['y1', 'y2']])
    
    # PCA
    pca = PCA(n_components=1)
    y_pca = pca.fit_transform(y_scaled)
    
    # MinMaxScaler 대신 QuantileTransformer 적용
    qt = QuantileTransformer(output_distribution='uniform', random_state=42)
    df['y_final'] = qt.fit_transform(y_pca)
    
    print(f"=== {name} ===")
    print(df['y_final'].describe())
    print(f"  0.0~0.3 구간: {(df['y_final'] < 0.3).sum()}개 ({(df['y_final'] < 0.3).mean()*100:.1f}%)")
    print(f"  0.3~0.7 구간: {((df['y_final'] >= 0.3) & (df['y_final'] < 0.7)).sum()}개 ({((df['y_final'] >= 0.3) & (df['y_final'] < 0.7)).mean()*100:.1f}%)")
    print(f"  0.7~1.0 구간: {(df['y_final'] >= 0.7).sum()}개 ({(df['y_final'] >= 0.7).mean()*100:.1f}%)")
    print()
    
    df.to_csv(rf'C:\workspace\finalproject\data\model_data_v1\model_{name}_final.csv', index=False)

print("완료!")

=== cleanser ===
count    1234.000000
mean        0.499999
std         0.289028
min         0.000000
25%         0.249691
50%         0.499962
75%         0.749808
max         1.000000
Name: y_final, dtype: float64
  0.0~0.3 구간: 371개 (30.1%)
  0.3~0.7 구간: 493개 (40.0%)
  0.7~1.0 구간: 370개 (30.0%)

=== mask ===
count    1310.000000
mean        0.499999
std         0.289002
min         0.000000
25%         0.249736
50%         0.499996
75%         0.749920
max         1.000000
Name: y_final, dtype: float64
  0.0~0.3 구간: 393개 (30.0%)
  0.3~0.7 구간: 524개 (40.0%)
  0.7~1.0 구간: 393개 (30.0%)

=== skincare ===
count    4534.000000
mean        0.499997
std         0.288765
min         0.000000
25%         0.249669
50%         0.500116
75%         0.749793
max         1.000000
Name: y_final, dtype: float64
  0.0~0.3 구간: 1360개 (30.0%)
  0.3~0.7 구간: 1814개 (40.0%)
  0.7~1.0 구간: 1360개 (30.0%)

=== suncare ===
count    396.000000
mean       0.500000
std        0.289771
min        0.000000
25%        0.2

In [4]:
import pandas as pd

files = ['cleanser', 'mask', 'skincare', 'suncare']

for name in files:
    df = pd.read_csv(rf'C:\workspace\finalproject\data\model_data_v1\model_{name}_final.csv')
    
    # cleaned.csv도 y_final 업데이트
    cleaned = pd.read_csv(rf'C:\workspace\finalproject\data\model_data_v1\model_{name}_cleaned.csv')
    cleaned['y_final'] = df['y_final']
    cleaned.to_csv(rf'C:\workspace\finalproject\data\model_data_v1\model_{name}_cleaned.csv', index=False)
    
    # y.csv 다시 저장
    df['y_final'].to_csv(rf'C:\workspace\finalproject\data\model_data_v1\model_{name}_y.csv', index=False)
    print(f"[{name}] y.csv 업데이트 완료")

print("완료!")

[cleanser] y.csv 업데이트 완료
[mask] y.csv 업데이트 완료
[skincare] y.csv 업데이트 완료
[suncare] y.csv 업데이트 완료
완료!


In [6]:
import pandas as pd

files = ['cleanser', 'mask', 'skincare', 'suncare']

for name in files:
    # cleaned.csv 기준으로 y값 다시 맞추기
    cleaned = pd.read_csv(rf'C:\workspace\finalproject\data\model_data_v1\model_{name}_cleaned.csv')
    
    # y.csv 다시 저장 (cleaned 기준)
    cleaned['y_final'].to_csv(
        rf'C:\workspace\finalproject\data\model_data_v1\model_{name}_y.csv', 
        index=False
    )
    
    # X랑 y 행 수 확인
    X = pd.read_csv(rf'C:\workspace\finalproject\data\model_data_v1\model_{name}_X.csv')
    y = cleaned['y_final']
    print(f"[{name}] X: {len(X)}행, y: {len(y)}행 {'✅' if len(X)==len(y) else '❌'}")

print("완료!")

[cleanser] X: 1212행, y: 1212행 ✅
[mask] X: 1226행, y: 1226행 ✅
[skincare] X: 4412행, y: 4412행 ✅
[suncare] X: 396행, y: 396행 ✅
완료!


In [8]:
import pandas as pd

df = pd.read_csv(r"C:\workspace\finalproject\data\model_data_v1\model_cleanser_cleaned.csv")
print(df['y_final'].describe())
print(f"0.0~0.3: {(df['y_final'] < 0.3).mean()*100:.1f}%")
print(f"0.3~0.7: {((df['y_final'] >= 0.3) & (df['y_final'] < 0.7)).mean()*100:.1f}%")
print(f"0.7~1.0: {(df['y_final'] >= 0.7).mean()*100:.1f}%")

count    1212.000000
mean        0.501344
std         0.288965
min         0.000000
25%         0.252841
50%         0.502377
75%         0.750174
max         1.000000
Name: y_final, dtype: float64
0.0~0.3: 29.9%
0.3~0.7: 40.0%
0.7~1.0: 30.1%


In [9]:
import pandas as pd

files = ['cleanser', 'mask', 'skincare', 'suncare']

for name in files:
    cleaned = pd.read_csv(rf'C:\workspace\finalproject\data\model_data_v1\model_{name}_cleaned.csv')
    
    cleaned['y_final'].to_csv(
        rf'C:\workspace\finalproject\data\model_data_v1\model_{name}_y.csv', 
        index=False
    )
    
    X = pd.read_csv(rf'C:\workspace\finalproject\data\model_data_v1\model_{name}_X.csv')
    y = cleaned['y_final']
    print(f"[{name}] X: {len(X)}행, y: {len(y)}행 {'✅' if len(X)==len(y) else '❌'}")

print("완료!")

[cleanser] X: 1212행, y: 1212행 ✅
[mask] X: 1226행, y: 1226행 ✅
[skincare] X: 4412행, y: 4412행 ✅
[suncare] X: 396행, y: 396행 ✅
완료!


### 분류 모델로 학습

In [10]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

files = {
    "cleanser": r"C:\workspace\finalproject\data\model_data_v1\model_cleanser",
    "skincare": r"C:\workspace\finalproject\data\model_data_v1\model_skincare",
    "mask":     r"C:\workspace\finalproject\data\model_data_v1\model_mask",
    "suncare":  r"C:\workspace\finalproject\data\model_data_v1\model_suncare",
}

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "RandomForest":       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "XGBoost":            XGBClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbosity=0, n_jobs=-1),
    "LightGBM":           LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, n_jobs=-1, verbose=-1),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for cat, path in files.items():
    X = pd.read_csv(f"{path}_X.csv")
    y_raw = pd.read_csv(f"{path}_y.csv").squeeze()
    
    # 상위 30% → 2, 30~70% → 1, 하위 30% → 0
    y = pd.cut(y_raw,
               bins=[0, y_raw.quantile(0.3), y_raw.quantile(0.7), 1.0],
               labels=[0, 1, 2],
               include_lowest=True
    ).astype(int)
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    print(f"\n{'='*55}")
    print(f"[{cat.upper()}] 클래스 분포: 0={( y==0).sum()} / 1={(y==1).sum()} / 2={(y==2).sum()}")
    print(f"{'='*55}")
    print(f"{'모델':<22} {'Accuracy':>10} {'F1(macro)':>12}")
    print(f"{'-'*55}")
    
    for name, model in models.items():
        acc = cross_val_score(model, X_scaled, y, cv=skf, scoring='accuracy').mean()
        f1  = cross_val_score(model, X_scaled, y, cv=skf, scoring='f1_macro').mean()
        print(f"{name:<22} {acc:>10.4f} {f1:>12.4f}")


[CLEANSER] 클래스 분포: 0=364 / 1=484 / 2=364
모델                       Accuracy    F1(macro)
-------------------------------------------------------
LogisticRegression         0.3738       0.3242
RandomForest               0.3565       0.3289
XGBoost                    0.3663       0.3342
LightGBM                   0.3457       0.3340

[SKINCARE] 클래스 분포: 0=1324 / 1=1764 / 2=1324
모델                       Accuracy    F1(macro)
-------------------------------------------------------
LogisticRegression         0.3760       0.2627
RandomForest               0.3540       0.3369
XGBoost                    0.3604       0.2884
LightGBM                   0.3565       0.3254

[MASK] 클래스 분포: 0=368 / 1=490 / 2=368
모델                       Accuracy    F1(macro)
-------------------------------------------------------
LogisticRegression         0.3776       0.3470
RandomForest               0.3589       0.3360
XGBoost                    0.3654       0.3328
LightGBM                   0.3532       0.3385

[

### price 피처 추가

In [11]:
import pandas as pd

df = pd.read_csv(r"C:\workspace\finalproject\data\model_data_v1\model_cleanser_cleaned.csv")
print("price" in df.columns)
print(df.columns.tolist())

False
['parent_asin', 'title', 'item_form', 'skin_type', 'material_type_free', 'product_benefits', 'average_rating', 'rating_number', 'y2', 'model', 'y1', 'y_final', 'title_clean', 'item_form_norm', 'skin_type_norm', 'material_norm', 'benefit_norm', 'skin_type_clean', 'material_clean', 'benefits_clean']


### suncare RandomForest 튜닝

In [12]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
import numpy as np

path = r"C:\workspace\finalproject\data\model_data_v1\model_suncare"
X = pd.read_csv(f"{path}_X.csv")
y_raw = pd.read_csv(f"{path}_y.csv").squeeze()

y = pd.cut(y_raw,
           bins=[0, y_raw.quantile(0.3), y_raw.quantile(0.7), 1.0],
           labels=[0, 1, 2],
           include_lowest=True
).astype(int)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

param_grid = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', None],
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_grid,
    n_iter=50,
    cv=skf,
    scoring='f1_macro',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(X_scaled, y)

print(f"\n최적 파라미터: {search.best_params_}")
print(f"튜닝 전 F1: 0.4334")
print(f"튜닝 후 F1: {search.best_score_:.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits

최적 파라미터: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 15}
튜닝 전 F1: 0.4334
튜닝 후 F1: 0.4358


In [13]:
import pandas as pd

df = pd.read_csv(r"C:\workspace\finalproject\data\model_data_v1\model_cleanser_cleaned.csv")
print(df['title_clean'].head(10).to_string())
print(f"\n결측치: {df['title_clean'].isnull().sum()}")

0    om botanical one step face cleanser allinone v...
1    cosmedix crystal cleanse hydrating liquid crys...
2    create cosmetics 5 elements cleanser 2 glycoli...
3    versed gentle cycle hydrating milky cleanser l...
4    mad hippie cream cleanser hydrating facial cle...
5    skinkick blemish relief 30 day kit clears prob...
6    kidskin gentle skin cleanser facial cleanser f...
7    brillare real rose powder facial cleanser mild...
8    hydroponic skincare marlo bloom lavish whipped...
9    pacifica beauty cactus water micellar cleansin...

결측치: 0


In [14]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix
import warnings
warnings.filterwarnings('ignore')

files = {
    "cleanser": r"C:\workspace\finalproject\data\model_data_v1\model_cleanser",
    "skincare": r"C:\workspace\finalproject\data\model_data_v1\model_skincare",
    "mask":     r"C:\workspace\finalproject\data\model_data_v1\model_mask",
    "suncare":  r"C:\workspace\finalproject\data\model_data_v1\model_suncare",
}

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "RandomForest":       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "XGBoost":            XGBClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbosity=0, n_jobs=-1),
    "LightGBM":           LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, n_jobs=-1, verbose=-1),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for cat, path in files.items():
    X = pd.read_csv(f"{path}_X.csv")
    y_raw = pd.read_csv(f"{path}_y.csv").squeeze()
    cleaned = pd.read_csv(f"{path}_cleaned.csv")
    
    y = pd.cut(y_raw,
               bins=[0, y_raw.quantile(0.3), y_raw.quantile(0.7), 1.0],
               labels=[0, 1, 2],
               include_lowest=True
    ).astype(int)
    
    # TF-IDF (상위 100개 키워드)
    tfidf = TfidfVectorizer(max_features=100, ngram_range=(1,2), stop_words='english')
    title_tfidf = tfidf.fit_transform(cleaned['title_clean'].fillna(''))
    
    # 기존 피처 + TF-IDF 합치기
    scaler = StandardScaler()
    X_scaled = csr_matrix(scaler.fit_transform(X))
    X_combined = hstack([X_scaled, title_tfidf])
    
    print(f"\n{'='*55}")
    print(f"[{cat.upper()}] 피처 수: 기존 {X.shape[1]}개 + TF-IDF 100개 = {X_combined.shape[1]}개")
    print(f"{'='*55}")
    print(f"{'모델':<22} {'Accuracy':>10} {'F1(macro)':>12}")
    print(f"{'-'*55}")
    
    for name, model in models.items():
        acc = cross_val_score(model, X_combined, y, cv=skf, scoring='accuracy').mean()
        f1  = cross_val_score(model, X_combined, y, cv=skf, scoring='f1_macro').mean()
        print(f"{name:<22} {acc:>10.4f} {f1:>12.4f}")


[CLEANSER] 피처 수: 기존 57개 + TF-IDF 100개 = 157개
모델                       Accuracy    F1(macro)
-------------------------------------------------------
LogisticRegression         0.3408       0.3155
RandomForest               0.3581       0.3168
XGBoost                    0.3276       0.3047
LightGBM                   0.3515       0.3317

[SKINCARE] 피처 수: 기존 67개 + TF-IDF 100개 = 167개
모델                       Accuracy    F1(macro)
-------------------------------------------------------
LogisticRegression         0.3570       0.3144
RandomForest               0.3586       0.3136


KeyboardInterrupt: 